# 🧪 Notebook 07 — Retrosynthesis Analysis
## ReactionRoute ML Suite
**Author**: Saleema Begam A | Cheminformatics Scientist

**Goal:** Predict retrosynthetic routes for top EGFR inhibitors using:
- **AiZynthFinder** (AstraZeneca) — MCTS-based retrosynthesis with neural network policy
- **ASKCOS** (MIT) — reaction condition prediction and retrosynthetic planning
- **IBM RXN** — transformer-based synthesis prediction

**Why this matters:**
- Identifies purchasable precursors for active EGFR compounds
- Maps synthetic routes to patent claim territories (Markush awareness)
- Bridges computational drug discovery with synthetic accessibility

**Patent context:**
- Erlotinib (Tarceva) — quinazoline core, aniline R-group
- Gefitinib (Iressa) — morpholine side chain, fluorine substituent
- Osimertinib (Tagrisso) — acrylamide covalent warhead, indole core

## STEP 1 — Install AiZynthFinder

In [ ]:
# Install AiZynthFinder — AstraZeneca open-source retrosynthesis tool
# Built on Monte Carlo Tree Search + neural network reaction templates

# Remove conflicting packages first
!pip uninstall -y numpy pandas scipy scikit-learn -q

# Install compatible scientific stack
!pip install numpy==1.26.4 pandas==2.2.2 scipy==1.11.4 scikit-learn==1.4.2 --quiet

# Install visualization libraries
!pip install matplotlib pillow psutil pyyaml --quiet

# Install AiZynthFinder (pinned for API stability)
!pip install aizynthfinder==4.3.0 --quiet

print("✅ Installation complete!")

In [ ]:
# Verify core imports
import gc
import os
import yaml
import subprocess
import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import psutil
from IPython.display import Image, display
from rdkit import Chem
from rdkit.Chem import Draw, AllChem, Descriptors, rdMolDescriptors
from rdkit.Chem.Scaffolds import MurckoScaffold

print(f"numpy  : {np.__version__}")
print(f"pandas : {pd.__version__}")
print("✅ All imports successful")

## STEP 2 — Download Pre-trained Models and Stock Database

In [ ]:
# Download AiZynthFinder public models
# - USPTO-trained neural network policy (reaction templates)
# - ZINC database stock collection (purchasable compounds)

DATA_DIR = "/content/aizynthfinder_data"
os.makedirs(DATA_DIR, exist_ok=True)

result = subprocess.run(
    ["python", "-m", "aizynthfinder.tools.download_public_data", DATA_DIR],
    capture_output=True, text=True
)
print("STDOUT:", result.stdout)
if result.stderr:
    print("STDERR:", result.stderr)

print("\n📁 Files in aizynthfinder_data/:")
for f in sorted(os.listdir(DATA_DIR)):
    size = os.path.getsize(os.path.join(DATA_DIR, f)) / 1e6
    print(f"  {f}  ({size:.1f} MB)")

In [ ]:
# Create config.yml — run this cell ONCE after download
DATA_DIR = "/content/aizynthfinder_data"

config = {
    "expansion": {
        "uspto": {
            "type": "template-based",
            "model": f"{DATA_DIR}/uspto_model.onnx",
            "template": f"{DATA_DIR}/uspto_templates.csv.gz"
        }
    },
    "filter": {
        "uspto_filter": {
            "type": "feasibility",
            "model": f"{DATA_DIR}/uspto_filter_model.onnx"
        }
    },
    "stock": {
        "zinc": f"{DATA_DIR}/zinc_stock.hdf5"
    },
    "search": {
        "iteration_limit": 100,
        "time_limit": 120
    }
}

CONFIG_PATH = f"{DATA_DIR}/config.yml"
with open(CONFIG_PATH, "w") as f:
    yaml.dump(config, f)

print(f"✅ config.yml written to {CONFIG_PATH}")

## STEP 3 — Define EGFR Target Compounds
Top active EGFR inhibitors from our ADMET pipeline (pIC50 ≥ 7.0)

In [ ]:
# Top EGFR inhibitors — key Markush scaffold families
EGFR_COMPOUNDS = [
    {
        'name': 'Erlotinib (Tarceva)',
        'smiles': 'C#Cc1cccc(Nc2ncnc3cc(OCCO)c(OCCO)cc23)c1',
        'patent': 'US5747498',
        'scaffold': 'Quinazoline',
        'pIC50': 9.3
    },
    {
        'name': 'Gefitinib (Iressa)',
        'smiles': 'COc1cc2ncnc(Nc3ccc(F)c(Cl)c3)c2cc1OCCCN1CCOCC1',
        'patent': 'EP0566226',
        'scaffold': 'Quinazoline',
        'pIC50': 8.9
    },
    {
        'name': 'Osimertinib (Tagrisso)',
        'smiles': 'C=CC(=O)Nc1cc2c(Nc3ccc(F)c(Cl)c3)ncnc2cc1NC(C)c1ccccn1',
        'patent': 'WO2013014448',
        'scaffold': 'Acrylamide-Pyrimidine',
        'pIC50': 9.1
    },
    {
        'name': 'Lapatinib (Tykerb)',
        'smiles': 'Clc1ccc(Nc2ncnc3cc(OCC4CCCO4)c(OCC4CCCO4)cc23)cc1',
        'patent': 'WO9935146',
        'scaffold': 'Quinazoline',
        'pIC50': 8.6
    },
    {
        'name': 'Afatinib (Gilotrif)',
        'smiles': 'C=CC(=O)N1CCN(c2ccc(Nc3ncnc4cc(OC)c(OC)cc34)cc2F)CC1',
        'patent': 'WO2002050043',
        'scaffold': 'Acrylamide-Quinazoline',
        'pIC50': 9.0
    }
]

df_targets = pd.DataFrame(EGFR_COMPOUNDS)
print('✅ Target EGFR compounds loaded:')
print(df_targets[['name', 'scaffold', 'pIC50', 'patent']].to_string(index=False))

## STEP 4 — Visualize Target Compounds

In [ ]:
os.makedirs('results/figures', exist_ok=True)

mols    = [Chem.MolFromSmiles(c['smiles']) for c in EGFR_COMPOUNDS]
legends = [f"{c['name']}\npIC50 = {c['pIC50']}" for c in EGFR_COMPOUNDS]

img = Draw.MolsToGridImage(
    mols,
    molsPerRow=3,
    subImgSize=(400, 300),
    legends=legends
)

out_path = 'results/figures/egfr_target_compounds.png'
img.save(out_path)
display(img)
print(f'✅ Saved: {out_path}')

## STEP 5 — AiZynthFinder Retrosynthesis
Monte Carlo Tree Search (MCTS) breaks each molecule into purchasable precursors

In [ ]:
from aizynthfinder.aizynthfinder import AiZynthFinder

DATA_DIR   = "/content/aizynthfinder_data"
CONFIG_PATH = f"{DATA_DIR}/config.yml"
os.makedirs("results/metrics", exist_ok=True)

# Check RAM before loading models
ram = psutil.virtual_memory()
print(f"Available RAM : {ram.available / 1e9:.1f} GB")
print(f"Total RAM     : {ram.total / 1e9:.1f} GB")

# Initialise finder from config file (correct API)
finder = AiZynthFinder(configfile=CONFIG_PATH)
finder.expansion_policy.select("uspto")
finder.stock.select("zinc")

# Conservative limits to avoid Colab RAM crash
# Increase iteration_limit to 100+ on Colab Pro or local machine
finder.config.search.iteration_limit = 20
finder.config.search.time_limit      = 30

print("\n✅ AiZynthFinder initialised. Starting retrosynthesis...\n")

results = []

for compound in EGFR_COMPOUNDS:
    print(f"Analyzing : {compound['name']}")
    print(f"Patent    : {compound['patent']} | Scaffold: {compound['scaffold']}")
    print("-" * 60)

    try:
        finder.target_smiles = compound["smiles"]
        finder.tree_search()
        finder.build_routes()

        n_routes = len(finder.routes)
        print(f"  Routes found : {n_routes}")

        if n_routes > 0:
            # Correct API: routes[i] is a RouteCollection entry dict
            top_route   = finder.routes[0]
            rt          = top_route["reaction_tree"]
            # depth() is a method; leaf molecules via .molecules()
            steps       = rt.depth
            leaf_mols   = [node for node in rt.graph.nodes
                           if rt.graph.out_degree(node) == 0]
            precursors  = [getattr(m, 'smiles', str(m))
                           for m in leaf_mols[:3]]
        else:
            steps, precursors = 0, []

        print(f"  Top route steps  : {steps}")
        print(f"  Key precursors   : {precursors}")

    except Exception as e:
        print(f"  ⚠️  Search error: {e}")
        n_routes, steps, precursors = 0, 0, []

    results.append({
        "compound"       : compound["name"],
        "patent"         : compound["patent"],
        "scaffold"       : compound["scaffold"],
        "pIC50"          : compound["pIC50"],
        "routes_found"   : n_routes,
        "top_route_steps": steps,
        "key_precursors" : str(precursors)
    })

    # Checkpoint after every compound
    pd.DataFrame(results).to_csv(
        "results/metrics/retrosynthesis_results.csv", index=False
    )
    print("  ✅ Checkpoint saved\n")

    # Free MCTS tree memory before next compound
    del finder._search_context
    gc.collect()

df_results = pd.DataFrame(results)
print("\n✅ Retrosynthesis complete!\n")
print(df_results[["compound", "routes_found", "top_route_steps"]])

## STEP 6 — Visualize Retrosynthetic Routes

In [ ]:
import gc
os.makedirs("results/figures", exist_ok=True)

# Re-run Erlotinib (reuse already-loaded finder)
finder.config.search.iteration_limit = 20
finder.config.search.time_limit      = 30
finder.target_smiles = EGFR_COMPOUNDS[0]["smiles"]  # Erlotinib
finder.tree_search()
finder.build_routes()

if len(finder.routes) > 0:
    # Correct API: routes[0] is a dict; image is accessed via .image property
    route_img = finder.routes[0].image
    out_path  = "results/figures/erlotinib_retrosynthesis_route.png"
    route_img.save(out_path)
    display(route_img)
    print(f"Total routes found : {len(finder.routes)}")
    print(f"✅ Saved: {out_path}")
else:
    print("No routes found — try increasing iteration_limit on Colab Pro")

del finder._search_context
gc.collect()

## STEP 7 — Patent-Aware Synthetic Accessibility Analysis
Connecting retrosynthesis with the Markush patent landscape

In [ ]:
def synthetic_accessibility_profile(compound):
    """
    Build patent-aware synthetic accessibility profile.
    Combines RDKit descriptors with patent context.

    In pharmaceutical patent analysis, synthetic routes define:
    - Which precursors are commercially available vs proprietary
    - Which reaction steps fall within existing process patents
    - Which intermediates are covered by Markush claims
    """
    mol = Chem.MolFromSmiles(compound['smiles'])
    if mol is None:
        return None

    scaffold        = MurckoScaffold.GetScaffoldForMol(mol)
    scaffold_smiles = Chem.MolToSmiles(scaffold)

    mw       = Descriptors.MolWt(mol)
    logp     = Descriptors.MolLogP(mol)
    hbd      = rdMolDescriptors.CalcNumHBD(mol)
    hba      = rdMolDescriptors.CalcNumHBA(mol)
    tpsa     = Descriptors.TPSA(mol)
    rotbonds = rdMolDescriptors.CalcNumRotatableBonds(mol)
    rings    = rdMolDescriptors.CalcNumRings(mol)

    patterns = {
        'quinazoline_core'  : Chem.MolFromSmarts('c1cnc2ccccc2n1'),
        'acrylamide_warhead': Chem.MolFromSmarts('C=CC(=O)N'),
        'aniline_rgroup'    : Chem.MolFromSmarts('c1ccc(N)cc1'),
        'fluorine_sub'      : Chem.MolFromSmarts('[F]c1ccccc1'),
    }
    pharm_flags = {
        name: mol.HasSubstructMatch(pat)
        for name, pat in patterns.items() if pat
    }

    return {
        'compound'     : compound['name'],
        'patent'       : compound['patent'],
        'scaffold'     : scaffold_smiles,
        'MW'           : round(mw, 2),
        'LogP'         : round(logp, 2),
        'HBD'          : hbd,
        'HBA'          : hba,
        'TPSA'         : round(tpsa, 2),
        'RotBonds'     : rotbonds,
        'Rings'        : rings,
        'Lipinski_pass': all([mw <= 500, logp <= 5, hbd <= 5, hba <= 10]),
        **pharm_flags
    }

profiles   = [synthetic_accessibility_profile(c) for c in EGFR_COMPOUNDS]
df_profiles = pd.DataFrame(profiles)

print('✅ Patent-aware synthetic accessibility profiles:')
print(df_profiles[[
    'compound', 'MW', 'LogP', 'TPSA',
    'Lipinski_pass', 'quinazoline_core', 'acrylamide_warhead'
]].to_string(index=False))

## STEP 8 — Scaffold Comparison Across Patent Families

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1 — pIC50 by scaffold family
scaffold_groups = df_targets.groupby('scaffold')['pIC50'].mean().sort_values(ascending=False)
colors = ['#2E75B6' if 'Acrylamide' in s else '#7F77DD' for s in scaffold_groups.index]
axes[0].barh(scaffold_groups.index, scaffold_groups.values, color=colors)
axes[0].set_title('Mean pIC50 by Scaffold Family', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Mean pIC50')
axes[0].set_xlim(8, 10)
for i, v in enumerate(scaffold_groups.values):
    axes[0].text(v + 0.02, i, f'{v:.2f}', va='center', fontsize=11)

# Plot 2 — Normalised physicochemical properties
props       = ['MW', 'LogP', 'TPSA', 'RotBonds']
x           = np.arange(len(props))
width       = 0.15
colors_list = ['#2E75B6', '#7F77DD', '#1D9E75', '#D85A30', '#BA7517']

for i, (_, row) in enumerate(df_profiles.iterrows()):
    vals      = [row[p] for p in props]
    norm_vals = [vals[0]/500, vals[1]/5, vals[2]/140, vals[3]/10]
    axes[1].bar(x + i*width, norm_vals, width,
                label=row['compound'].split(' ')[0],
                color=colors_list[i], alpha=0.85)

axes[1].set_xticks(x + width*2)
axes[1].set_xticklabels(['MW/500', 'LogP/5', 'TPSA/140', 'RotBonds/10'])
axes[1].set_title('Normalised Physicochemical Properties', fontsize=13, fontweight='bold')
axes[1].axhline(y=1.0, color='red', linestyle='--', alpha=0.5, label='Lipinski limit')
axes[1].legend(fontsize=8)

plt.tight_layout()
plt.savefig('results/figures/retrosynthesis_scaffold_analysis.png', dpi=150)
plt.show()
print('✅ Saved: results/figures/retrosynthesis_scaffold_analysis.png')

## STEP 9 — Save All Results

In [ ]:
os.makedirs('results/metrics', exist_ok=True)

# Graceful fallback if Step 5 was skipped
if 'df_results' not in dir():
    df_results = pd.DataFrame(EGFR_COMPOUNDS)[['name','patent','scaffold','pIC50']]
    df_results['routes_found']    = 'Step 5 not run'
    df_results['top_route_steps'] = 'Step 5 not run'

df_results.to_csv('results/metrics/retrosynthesis_results.csv',    index=False)
df_profiles.to_csv('results/metrics/egfr_synthetic_profiles.csv',  index=False)

print('✅ All results saved!')
print('\nFiles:')
for p in [
    'results/metrics/retrosynthesis_results.csv',
    'results/metrics/egfr_synthetic_profiles.csv',
    'results/figures/egfr_target_compounds.png',
    'results/figures/erlotinib_retrosynthesis_route.png',
    'results/figures/retrosynthesis_scaffold_analysis.png',
]:
    exists = '✅' if os.path.exists(p) else '⏳ (not yet generated)'
    print(f'  {exists}  {p}')

print('\n✅ Notebook 07 complete!')

## STEP 10 — ASKCOS (MIT) — Alternative Retrosynthesis
Web-based tool — no installation needed. Free at https://askcos.mit.edu

In [ ]:
def askcos_retrosynthesis(smiles, max_depth=3):
    """
    Query ASKCOS MIT API for retrosynthetic planning.
    Returns predicted reaction conditions and precursors.
    Fallback: manual use at https://askcos.mit.edu
    """
    url     = 'https://askcos.mit.edu/api/v2/tree-builder/'
    payload = {
        'smiles'        : smiles,
        'max_depth'     : max_depth,
        'max_branching' : 25,
        'expansion_time': 30,
        'max_trees'     : 5,
        'store_results' : True
    }
    try:
        response = requests.post(url, json=payload, timeout=60)
        if response.status_code == 200:
            return response.json()
        else:
            return {'error': f'HTTP {response.status_code}'}
    except Exception as e:
        return {'error': str(e)}

erlotinib_smiles = EGFR_COMPOUNDS[0]['smiles']
print('Querying ASKCOS MIT for Erlotinib retrosynthesis...')
result = askcos_retrosynthesis(erlotinib_smiles)

if 'error' not in result:
    print('✅ ASKCOS result received!')
    print(f"Task ID: {result.get('task_id', 'N/A')}")
else:
    print(f"ℹ️  ASKCOS API note: {result['error']}")
    print('\n👉 Run manually at: https://askcos.mit.edu')
    print(f'   Paste SMILES: {erlotinib_smiles}')